# RG3 12개 모델 비라벨 적용


In [1]:
from pathlib import Path
import json,nbformat
import numpy as np,pandas as pd
root=Path.cwd();tables=root/'reports/tables'
s10=''.join(c.source for c in nbformat.read('notebooks/10_original_defect_report_revision.ipynb',4).cells if c.cell_type=='code')
pre={'__name__':'__main__'}
exec(s10.split('shots = label_shots.merge')[0],pre)
observed=pre['observed'];keys=pre['keys'];raw_features=pre['features']
saved=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
joined=saved.merge(observed,on=keys,suffixes=('_saved','_rebuilt'),validate='one_to_one')
for c in [c for c in saved if c in observed and c not in keys+['history_segment','sensor_conflict']]:
 assert np.allclose(joined[c+'_saved'].to_numpy(float),joined[c+'_rebuilt'].to_numpy(float),rtol=1e-8,atol=1e-7,equal_nan=True),c
u=pd.read_csv(root/'data/original_split/moldset_unlabeled_rg3.csv',parse_dates=['TimeStamp'])
ukeys=pd.MultiIndex.from_frame(u.assign(product='RG3')[keys]).unique()
known=pd.read_csv(root/'data/original_split/moldset_labeled_rg3.csv',parse_dates=['TimeStamp'])
knownkeys=pd.MultiIndex.from_frame(known.assign(product='RG3')[keys]).unique()
extra=pd.read_csv(root/'data/original/labeled_data.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp'],parse_dates=['TimeStamp'])
extra=extra[extra.PART_NAME.str.contains('RG3',na=False)].assign(product='RG3')
knownkeys=knownkeys.union(pd.MultiIndex.from_frame(extra[keys]).unique())
qtable=observed[pd.MultiIndex.from_frame(observed[keys]).isin(ukeys)&observed['product'].eq('RG3')].copy()
overlap=pd.MultiIndex.from_frame(qtable[keys]).isin(knownkeys)
overlap_count=int(overlap.sum());qtable=qtable[~overlap].copy()
source=''.join(c.source for c in nbformat.read('notebooks/193_rg3_fresh12_full_reproduction.ipynb',4).cells if c.cell_type=='code')
ns={'__name__':'__main__'}
exec(source[:source.index('records=[];cutrows=[];first=None')],ns)
train=ns['d'].copy()
qtable=qtable[qtable.EQUIP_CD.isin(train.EQUIP_CD.unique())].reset_index(drop=True)
# Same required columns as saved gas inference, no fabricated history for incomplete rows.
s154=nbformat.read('notebooks/154_rg3_gas_model_export.ipynb',4).cells[1].source
schema={'__name__':'__main__'}
exec(s154[:s154.index('def normalize_raw(')],schema)
required=schema['NEEDED']
ready=np.isfinite(qtable[required].to_numpy(float)).all(axis=1)&~qtable.sensor_conflict.fillna(True).to_numpy()
out=qtable[keys+['history_segment','sensor_conflict','Barrel_Temperature_1']].copy()
out['history_ready']=ready;out['gas_candidate']=np.nan
queries=qtable.loc[ready].reindex(columns=train.columns).copy()
queries['comparison_block']='UNLABELED_'+qtable.loc[ready,'history_segment'].astype(str)
queries['PassOrFail']=-1;queries['Reason']='UNKNOWN'
ns['d']=pd.concat([train,queries],ignore_index=True)
ns['y']=np.r_[train.PassOrFail.to_numpy(int),np.full(len(queries),-1)]
ns['groups']=ns['d'].comparison_block
ns['F']={k:pd.DataFrame(ns['physical'](ns['d'],k)) for k in ['thermal','pressure']}
ref=np.arange(len(train));q=np.arange(len(train),len(ns['d']))
assert set(ns['d'].TimeStamp.iloc[ref]).isdisjoint(ns['d'].TimeStamp.iloc[q])
cal=pd.read_csv(tables/'rg3_193_fresh12_predictions.csv',parse_dates=['TimeStamp']).set_index('TimeStamp').loc[train.TimeStamp]
normal=train.PassOrFail.eq(0).to_numpy()
cuts={k:float(np.quantile(cal.loc[normal,k+'_score'],quant,method='higher')) for k,quant in [('baseline',.99),('expert',.995),('thermal',.975),('pressure',.99)]}
delta=train.Barrel_Temperature_5__delta1.to_numpy()
cuts['delta']=float(np.quantile(delta[normal&np.isfinite(delta)],.95,method='higher'))
zcal=[]
for held in train.comparison_block.unique():
 ii=np.flatnonzero(train.comparison_block.eq(held))
 rr=np.flatnonzero(~train.comparison_block.eq(held))
 z=ns['local_residual'](rr,ii,'thermal')[:,4]
 zcal.extend(zip(ii,z))
zcal=np.array([v for _,v in sorted(zcal)])
cuts['thermal_gate']=float(np.quantile(zcal[normal],.05))
print('Scoring fresh12 on',len(q),'ready unknown shots; overlap excluded',overlap_count,flush=True)
scores,z=ns['fresh_scores'](ref,q)
query=ns['d'].iloc[q]
b=scores['baseline']>cuts['baseline']
e=(scores['expert']>cuts['expert'])&(query.Barrel_Temperature_5__delta1.to_numpy()>cuts['delta'])
t=(scores['thermal']>cuts['thermal'])&(z['thermal'][:,4]<cuts['thermal_gate'])
direction=((query.Max_Injection_Pressure__delta1.round(2)<0)&(query.Max_Injection_Speed__delta1.round(2)>0)&(query.Max_Switch_Over_Pressure__delta1.round(2)<query.Max_Injection_Pressure__delta1.round(2))).to_numpy()
p=(scores['pressure']>cuts['pressure'])&direction
out.loc[ready,'gas_candidate']=(b|e|t|p).astype(int)
for name,arr in [('baseline',b),('expert',e),('thermal',t),('pressure',p)]:
 out.loc[ready,name+'_candidate']=arr.astype(int)
 out.loc[ready,name+'_score']=scores[name]
out['operating_proxy']=out.Barrel_Temperature_1.ge(100)
out['date']=out.TimeStamp.dt.strftime('%Y-%m-%d')
out.to_csv(tables/'rg3_194_fresh12_unlabeled_predictions.csv',index=False,encoding='utf-8-sig')
out[out.gas_candidate.eq(1)].to_csv(tables/'rg3_194_fresh12_unlabeled_candidates.csv',index=False,encoding='utf-8-sig')
byday=out.groupby('date').agg(shots=('TimeStamp','size'),ready=('history_ready','sum'),candidates=('gas_candidate','sum'))
byday['alarm_percent']=100*byday.candidates/byday.ready.replace(0,np.nan)
byday.to_csv(tables/'rg3_194_fresh12_unlabeled_by_day.csv',encoding='utf-8-sig')
rows=[]
for cohort,mask in [('all_unknown',np.ones(len(out),bool)),('operating_proxy',out.operating_proxy.to_numpy()),('nonoperating_proxy',~out.operating_proxy.to_numpy())]:
 a=out[mask];n=int(a.history_ready.sum());count=int(a.gas_candidate.eq(1).sum())
 rows.append(dict(cohort=cohort,shots=len(a),ready=n,abstained=len(a)-n,candidates=count,alarm_percent=100*count/n if n else None))
summary={'raw_unlabeled_rows':len(u),'raw_unlabeled_unique_shots':len(ukeys),'known_shots_excluded':overlap_count,'training_normal':566,'training_gas':17,'classifiers':12,'counts':rows,'cuts':cuts,'interpretation':'gas candidates, unknown truth; not verified defects or accuracy','calibration':'frozen labeled OOF normal scores, no unlabeled labels or alarm-rate targeting','operating_proxy':'Barrel_Temperature_1 >=100, heuristic not actual operation labels'}
(tables/'rg3_194_fresh12_unlabeled_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
text='# RG3 12개 모델 비라벨 적용\n\n'
text+=pd.DataFrame(rows).to_string(index=False)+'\n\n'
text+='original_split 비라벨 RG3의 센서 이력을10번과동일하게재구성하고라벨1304샷피처일치를검사했다. 모든제공라벨파일과겹치는키는제외했다. 공식정상566/가스17만학습하고추가19개가스는사용하지않았다.12개는193과동일정의로새로학습했다.임계값은193의라벨구간제외정상점수로계산해고정,비라벨분포에맞춰조정하지않았다.\n\n'
text+='탐지는 가스 후보 경보이며실제불량확인이아니다.비라벨정답없어서TP/FP/F1계산불가.샷=설비/제품/timestamp로중복행을묶은단위.인과이력부족또는센서충돌은판정보류.작동구간구분은배럴1>=100도휴리스틱이며실제동작라벨이아니다.기존개발검증모델의전체재학습적용,독립검증아님.모델파일교체및commit/push없음.\n'
(root/'reports/rg3_194_fresh12_unlabeled_result.md').write_text(text,encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('Branches',out[[k+'_candidate' for k in ['baseline','expert','thermal','pressure']]].sum().to_dict())



Scoring fresh12 on 28710 ready unknown shots; overlap excluded 387


C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\nbformat\validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


{
  "raw_unlabeled_rows": 72484,
  "raw_unlabeled_unique_shots": 66145,
  "known_shots_excluded": 387,
  "training_normal": 566,
  "training_gas": 17,
  "classifiers": 12,
  "counts": [
    {
      "cohort": "all_unknown",
      "shots": 29236,
      "ready": 28710,
      "abstained": 526,
      "candidates": 734,
      "alarm_percent": 2.5566004876349706
    },
    {
      "cohort": "operating_proxy",
      "shots": 16069,
      "ready": 15797,
      "abstained": 272,
      "candidates": 731,
      "alarm_percent": 4.6274609102994235
    },
    {
      "cohort": "nonoperating_proxy",
      "shots": 13167,
      "ready": 12913,
      "abstained": 254,
      "candidates": 3,
      "alarm_percent": 0.023232401455897157
    }
  ],
  "cuts": {
    "baseline": 0.5063582715704156,
    "expert": 0.2904605070421565,
    "thermal": 0.5897268810636868,
    "pressure": 0.6646442905431882,
    "delta": 0.70001220703125,
    "thermal_gate": -1.8340121577667325
  },
  "interpretation": "gas candidat